In [ ]:
import time
import requests
import json

# enumerable for query types
class QueryType:
  BASIC = "cql"
  """ Basic query type for simple full-text searches using CQL (Corpus Query Language). This type allows you to perform straightforward searches based on specific terms or patterns in the text. It is suitable for users who want to retrieve records without complex query structures."""
  CORPUS = "fcs"
  """ Corpus query type for advanced searches using FCS (Full-text Corpus Search). This type enables users to perform more complex searches, including queries that involve multiple conditions, logical operators, and specific corpus structures. It is designed for users who require detailed and precise search capabilities within the text corpus."""
  LEXICAL = "lex"
  """ Lexical query type for searches in lexical resources. This type allows users to search for records that match specific lexical criteria, such as word forms, lemmas, or morphological patterns. It is intended for users who need to analyze the text at a lexical level and retrieve records that meet specific linguistic characteristics."""

class BaseClass:
  numberOfResults = 10
  """Default number of results to retrieve per resource."""
      
  def processJsonSearchResponse(self, url, response, removeEmptyRessources=True):
    """ Process the JSON response from a search query and extract relevant information about the resources and their records.
    Parameters:
    - url (str): The URL used to request the records for the resource from the Text+-Aggregator. This URL is constructed based on the resource handle and other parameters.
    - response (requests.Response): The HTTP response object returned from the search query request. This object contains the JSON data with information about the resources and their records. 
    - removeEmptyRessources (bool): If True, resources with zero records will be excluded from the results. If False, all resources will be included in the results, even if they have no records.
    Returns:
    - list: A list of Result objects containing the search results. Each Result object includes information about the resource, number of records, and the actual records retrieved.    
    """
    tmp = json.loads(response.text)
            
    res = []                
    for r in tmp["results"]:
      if(removeEmptyRessources and r["numberOfRecords"] < 1):
        continue
      result = Result(url)
      result.resourceHandle = r["resourceHandle"]
      result.numberOfRecords = r["numberOfRecords"]
      result.requestUrl = r["requestUrl"]
      result.resource_endpointInstitution = r["resource"]["endpointInstitution"]["name"]
      result.resource_title = r["resource"]["title"]
      result.resource_languages = r["resource"]["languages"]
      result.records = r["records"]
      
      res.append(result)
    
    return res

class Result(BaseClass):
  __searchUrl = ""
  
  def __init__(self, searchUrl):
    """ Constructor for the Result class. Initializes the result object with the specified search URL, which is used to retrieve additional records for the resource if needed.
    Parameters:
    - searchUrl (str): The URL used to request the records for the resource from the Text+-Aggregator. This URL is constructed based on the resource handle and other parameters.
    """
    self.__searchUrl = searchUrl
  
  resourceHandle = ""
  """ The unique identifier (handle) of the resource in the Text+-Aggregator. This handle is used to reference the specific resource when retrieving records or performing searches. """
  numberOfRecords = 0
  """ The total number of records available for the resource in the Text+-Aggregator. This value indicates how many records can be retrieved for the resource. """
  requestUrl = ""
  """ The URL used to request the records for the resource from the Text+-Aggregator. This URL is constructed based on the resource handle and other parameters. """
  resource_endpointInstitution = ""
  """ The name of the endpoint institution associated with the resource in the Text+-Aggregator. This information provides context about the source or provider of the resource. """
  resource_title = ""
  """ The title of the resource in the Text+-Aggregator. This title provides a human-readable name for the resource, making it easier to identify and reference. """
  resource_languages = []
  """ A list of languages supported by the resource in the Text+-Aggregator. Each language is represented as a dictionary containing information such as the language code and name. """  
  records = []
  """ A list of records retrieved for the resource from the Text+-Aggregator. Each record is represented as a dictionary containing information about the specific record, such as its content and metadata. """
  
  def hasMoreRecords(self):
    """ Method to check if there are more records available for the current resource in the Text+-Aggregator.
    """
    return self.numberOfRecords > len(self.records)
      
  def moreRecords(self, wait=1):
      """ Method to retrieve additional records for the current resource from the Text+-Aggregator.
      """
      if(not self.hasMoreRecords()):
        return
      payload = {
        'resourceId': self.resourceHandle,
        'numberOfResults': 10
      }
      headers = {
        'Accept': 'application/json, text/plain, */*',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
        'Content-Type': 'application/x-www-form-urlencoded'
      }
      
      requests.request("GET", self.__searchUrl, headers=headers, data=payload)
      while self.__searchInProgress():
        time.sleep(wait)
      self.__searchAddResults()
  
  def __searchInProgress(self):
      """ Private method to check if the search for additional records is still in progress for the current resource.
      """
      url = self.__searchUrl + "/metaonly"
      payload = {}
      headers = {
        'Accept': 'application/json, text/plain, */*',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
      }
      response = requests.request("GET", url, headers=headers, data=payload)
      tmp = json.loads(response.text)
      
      return tmp["inProgress"] == 1
  
  def __searchAddResults(self):
      """ Private method to retrieve and add additional records for the current resource from the Text+-Aggregator.
      """
      url = self.__searchUrl + "?resourceId=" + self.resourceHandle
      payload = {}
      headers = {
        'Accept': 'application/json, text/plain, */*',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
      }
      response = requests.request("GET", url, headers=headers, data=payload)
      tmp = self.processJsonSearchResponse(url, response, False)
      for r in tmp[0].records:
        self.records.append(r)

class TextPlusClient(BaseClass):    
    resources = None 
    """ List of available resources retrieved from the Text+-Aggregator. Each resource is represented as a dictionary containing information such as the resource handle, title, endpoint institution, and supported languages. """
    languages = None
    """ List of available languages retrieved from the Text+-Aggregator. Each language is represented as a dictionary containing information such as the language code and name. """
    isConnected = False
    """ Boolean indicating whether the client is successfully connected to the Text+-Aggregator. This is determined by checking if the resources and languages have been successfully loaded. """
    __handles = []
    __currentSearchMax = -1
    __currentSearchSession = None
  
    def __init__(self, base_url="https://fcs.text-plus.org"):
        """ Constructor for the TextPlusClient class. Initializes the client with the specified base URL of the Text+-Aggregator.
        """
        self.base_url = base_url + "/rest"
        self.__loadResources()
        self.__loadLanguages()
        self.__validateConnection()
    
    def __loadResources(self):
        """ Private method to load the available resources from the Text+-Aggregator.
        """
        url = self.base_url + "/resources"
        payload = {}
        headers = {
          'Accept': 'application/json, text/plain, */*',
          'Cache-Control': 'no-cache',
          'Pragma': 'no-cache'          
        }
        response = requests.request("GET", url, headers=headers, data=payload)
        self.resources = json.loads(response.text)
        
        for resource in self.resources:
            self.__handles.append(resource["handle"])
    
    def __loadLanguages(self):
        """ Private method to load the available languages from the Text+-Aggregator.
        """
        url = self.base_url + "/languages"
        payload = {}
        headers = {
          'Accept': 'application/json, text/plain, */*',
          'Cache-Control': 'no-cache',
          'Pragma': 'no-cache',
        }
        response = requests.request("GET", url, headers=headers, data=payload)
        self.languages = json.loads(response.text)
    
    def __validateConnection(self):
        """ Private method to validate the connection to the Text+-Aggregator. 
        This method checks if the resources and languages have been successfully loaded, indicating a valid connection.
        """
        self.isConnected = self.resources != None and self.languages != None
    
    def search_custom_new(self, query, queryType=QueryType.BASIC, language="mul", restrictedToResource=None):      
        """ Method to initiate a custom search query against the Text+-Aggregator. 
        This method starts the search process and returns immediately, allowing you to check the status of the search and retrieve results later.
        For a complete search workflow, you can use the `search` method, which combines initiating the search, waiting for it to complete, and retrieving the results.
        
        Parameters:
        - query (str): The search query string. e.g., "*party" for a wildcard search.
        - queryType (str): The type of query to perform. Options are "cql" for basic queries, "fcs" for corpus queries, and "lex" for lexical queries. You can use the QueryType class to specify the query type.
        - language (str): The language code for the search. Use "mul" for multilingual searches or specify a specific language code. You can retrieve available languages using the `languages` attribute of the TextPlusClient instance.
        - restrictedToResource (list, optional): A list of resource handles to restrict the search to specific resources. If None, the search will be performed across all available resources. You can retrieve available resources using the `resources` attribute of the TextPlusClient instance.
        """
        url = self.base_url + "/search"
       
        payload = {
          'query': query,
          'queryType': queryType,
          'language': language,
          'numberOfResults': self.numberOfResults
        }
                
        if(restrictedToResource == None):
          self.__currentSearchMax = len(self.__handles)
          payload["resourceIds[]"] = self.__handles
        else:
          self.__currentSearchMax = len(restrictedToResource)
          payload["resourceIds[]"] = restrictedToResource        
        
        headers = {
          'Accept': 'application/json, text/plain, */*',
          'Cache-Control': 'no-cache',
          'Pragma': 'no-cache',
          'Content-Type': 'application/x-www-form-urlencoded'
        }
        
        response = requests.request("POST", url, headers=headers, data=payload)
        self.__currentSearchSession = json.loads(response.text)
    
    def __searchStatus(self):
        """ Private method to check the status of the current search session. 
        
        Returns:
        - dict: A dictionary containing the total number of resources, the number of resources still in progress, and the number of completed resources.
        """
        url = self.base_url + "/search/" + self.__currentSearchSession + "/metaonly"
        payload = {}
        headers = {
          'Accept': 'application/json, text/plain, */*',
          'Cache-Control': 'no-cache',
          'Pragma': 'no-cache',
        }
        response = requests.request("GET", url, headers=headers, data=payload)
        tmp = json.loads(response.text)
        
        return { "total": self.__currentSearchMax, "inProgress": tmp["inProgress"], "completed": (self.__currentSearchMax - tmp["inProgress"]) }
    
    def search_custom_wait(self, wait=1):
      """ If you run a custom search (search_custom_new), you can use this method to wait for the search to complete before retrieving the results.
      
      Parameters:
      - wait (int): The number of seconds to wait between status checks. Default is 1 second.
      """ 
      while self.__searchStatus()["inProgress"] > 0:
        time.sleep(wait)
    
    def search_custom_getResults(self, removeEmptyRessources=True):
        """ If you run a custom search (search_custom_new), you can use this method to retrieve the results after the search has completed (search_custom_wait).
        
        Parameters:
        - removeEmptyRessources (bool): If True, resources with zero records will be excluded from the results. If False, all resources will be included in the results, even if they have no records.
        
        Returns:
        - list: A list of Result objects containing the search results. Each Result object includes information about the resource, number of records, and the actual records retrieved.
        """ 
        url = self.base_url + "/search/" + self.__currentSearchSession
        payload = {}
        headers = {
          'Accept': 'application/json, text/plain, */*',
          'Cache-Control': 'no-cache',
          'Pragma': 'no-cache',
        }
        response = requests.request("GET", url, headers=headers, data=payload)
        return self.processJsonSearchResponse(url, response, removeEmptyRessources)
    
    def search(self, query, queryType=QueryType.BASIC, language="mul", restrictedToResource=None):
        """ Execute a search query against the Text+-Aggregator and return the results. 
        This method combines the steps of initiating the search, waiting for it to complete, and retrieving the results.
        
        Parameters:
        - query (str): The search query string. e.g., "*party" for a wildcard search.
        - queryType (str): The type of query to perform. Options are "cql" for basic queries, "fcs" for corpus queries, and "lex" for lexical queries. You can use the QueryType class to specify the query type.
        - language (str): The language code for the search. Use "mul" for multilingual searches or specify a specific language code. You can retrieve available languages using the `languages` attribute of the TextPlusClient instance.
        - restrictedToResource (list, optional): A list of resource handles to restrict the search to specific resources. If None, the search will be performed across all available resources. You can retrieve available resources using the `resources` attribute of the TextPlusClient instance.
        
        Returns:
        - list: A list of Result objects containing the search results. Each Result object includes information about the resource, number of records, and the actual records retrieved.        
        """
        self.search_custom_new(query, queryType, language, restrictedToResource)
        self.search_custom_wait()
        return self.search_custom_getResults()

In [102]:
client = TextPlusClient()
results = client.search("*party", QueryType.BASIC, "mul")
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump([r.__dict__ for r in results], f, ensure_ascii=False, indent=4)
    

In [ ]:
client = TextPlusClient()
client.search_custom_new("*party", QueryType.BASIC, "mul")
client.search_custom_wait(1)
results = client.search_custom_getResults()

In [100]:
for r in results:
  if(r.numberOfRecords > 0):
    print("Endpoint Institution: " + r.resource_endpointInstitution["en"])
    print("Resource: " + r.resource_title["en"] + " (" + r.resourceHandle + ")")
    print("Number of Records: " + str(r.numberOfRecords))
    print("Languages: " + str(r.resource_languages))

Endpoint Institution: Berlin-Brandenburg Academy of Sciences and Humanities
Resource: Digital Dictionary of the German Language (hdl:21.11120/0000-000B-146F-3)
Number of Records: 32
Languages: ['deu']
Endpoint Institution: Berlin-Brandenburg Academy of Sciences and Humanities
Resource: Dictionary of the German Contemporary Language (hdl:21.11120/0000-000B-241F-B)
Number of Records: 2
Languages: ['deu']
Endpoint Institution: Berlin-Brandenburg Academy of Sciences and Humanities
Resource: WAHRIG German Dictionary (https://www.dwds.de/ns/lexfcs/wdw)
Number of Records: 6
Languages: ['deu']
Endpoint Institution: German National Library
Resource: Free online dissertations in social sciences (dnb:resource:2)
Number of Records: 10057
Languages: ['deu']
Endpoint Institution: Saxon Academy of Sciences and Humanities in Leipzig
Resource: Wortschatz Leipzig German (saw:ws_deu)
Number of Records: 1
Languages: ['deu']
Endpoint Institution: Saxon Academy of Sciences and Humanities in Leipzig
Resource